# Master Benchmark Pipeline — All 5 Methods on Synthetic Specular Dataset

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full training sweep sequentially for all **5 method configurations** on the 8 Synthetic Specular scenes inside the BOSCH server environment.

### 5 Method Sweeps Executed:
1. **`fastgs`**: FastGS baseline (`FastGS_backup_v2`)
2. **`3dgs`**: 3D Gaussian Splatting baseline (`gaussian-splatting_backup`)
3. **`spec-gaussians`**: Specular-Gaussians baseline (`Specular-Gaussians_backup_v2`)
4. **`spec-fastgs (config=False)`**: Spec-FastGS baseline without proposed modules (`spec-fastgs`)
5. **`spec-fastgs (config=True)`**: Spec-FastGS full proposed model (`spec-fastgs`)

### Workflow for Each Method:
1. **Prompt for `HF_TOKEN` ONCE** at startup and validate HuggingFace authentication.
2. Run batch training sweep on all 8 scenes (`ashtray`, `dishes`, `headphone`, `jupyter`, `lock`, `plane`, `record`, `teapot`).
3. Aggregate quantitative metrics (`PSNR`, `SSIM`, `LPIPS`, `Gaussian point count`).
4. Zip output results and upload to Hugging Face (`DiBiay/<repo-name>`).
5. **Auto-Clean**: Purge local zip and `./output/` folder after successful upload to conserve disk space.
6. Display consolidated master summary table comparing all 5 methods.

## c00 — Proxy & Upfront HuggingFace Authentication
Sets BOSCH proxy and prompts for your HuggingFace Token **ONCE upfront**. Validates access before any training begins.

In [ ]:
# ── Proxy Settings & Upfront HuggingFace Token Validation ─────────────────────────
import os
import sys
import getpass

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

# Check for existing token in environment or prompt user once
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    print('\nPlease enter your Hugging Face write token (will not be displayed as you type):')
    HF_TOKEN = getpass.getpass('HF Token: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN

try:
    import huggingface_hub
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    user_info = api.whoami()
    print(f"\n✅ Hugging Face Authentication Successful! Logged in as: {user_info.get('name', 'User')}")
except Exception as e:
    print(f"\n❌ Hugging Face Authentication Failed: {e}")
    print("Please re-run this cell with a valid write-permission HuggingFace token.")

## c01 — Environment & Dataset Verification
Verifies hardware detection, `thesis_env` kernel, and Synthetic Specular source dataset layout.

In [ ]:
# ── Environment & Dataset Verification ───────────────────────────────────────────
import os
import sys
import torch

print(f'Active Python      : {sys.executable}')
print(f'PyTorch Version    : {torch.__version__}')
print(f'CUDA Available     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name    : {torch.cuda.get_device_name(0)}')

DATA_ROOT = "/home/ghp4hc/datasets/datasets/synthetic_specular"
SYNTHETIC_SCENES = [
    "ashtray", "dishes", "headphone", "jupyter",
    "lock", "plane", "record", "teapot"
]

assert os.path.exists(DATA_ROOT), f"Dataset path not found at {DATA_ROOT}! Please check dataset installation."
print(f"\n✅ Dataset Root verified at: {DATA_ROOT}")

missing_scenes = []
for scene in SYNTHETIC_SCENES:
    scene_dir = os.path.join(DATA_ROOT, scene)
    if not os.path.isdir(scene_dir):
        missing_scenes.append(scene)

if missing_scenes:
    print(f"⚠️  Missing {len(missing_scenes)} scenes: {missing_scenes}")
else:
    print(f"✅ All {len(SYNTHETIC_SCENES)} synthetic specular scenes verified!")

## c02 — Sequential Execution Pipeline (5 Methods)
Executes each method sequentially, parses quantitative metrics, archives results, uploads to HuggingFace using pre-validated `HF_TOKEN`, and purges local outputs.

In [ ]:
# ── Sequential Benchmark Pipeline for 5 Methods ─────────────────────────────────
import subprocess
import os
import sys
import json
import shutil
from huggingface_hub import HfApi

HOME = os.path.expanduser('~')
BASE_REPO_DIR = '/home/ghp4hc/thesis-all' if os.path.isdir('/home/ghp4hc/thesis-all') else os.getcwd()

EXPERIMENTS = [
    {
        "name": "fastgs",
        "display_name": "FastGS Baseline",
        "repo_subdir": "FastGS_backup_v2",
        "script": "run_synthetic_specular.sh",
        "hf_repo_id": "DiBiay/fastgs-synthetic-specular-result",
        "log_name": "synthetic_specular_fastgs_run.log",
        "zip_name": "fastgs_output_synthetic_specular.zip",
    },
    {
        "name": "3dgs",
        "display_name": "3D Gaussian Splatting Baseline",
        "repo_subdir": "gaussian-splatting_backup",
        "script": "run_synthetic_specular.sh",
        "hf_repo_id": "DiBiay/3dgs-synthetic-specular-result",
        "log_name": "synthetic_specular_3dgs_run.log",
        "zip_name": "3dgs_output_synthetic_specular.zip",
    },
    {
        "name": "spec-gaussians",
        "display_name": "Specular-Gaussians Baseline",
        "repo_subdir": "Specular-Gaussians_backup_v2",
        "script": "run_synthetic_specular.sh",
        "hf_repo_id": "DiBiay/specular_gaussians-synthetic-specular-result",
        "log_name": "synthetic_specular_specular_gaussians_run.log",
        "zip_name": "specular_gaussians_output_synthetic_specular.zip",
    },
    {
        "name": "spec-fastgs-config-false",
        "display_name": "Spec-FastGS (config=False)",
        "repo_subdir": "spec-fastgs",
        "script": "run_synthetic_specular_config_false.sh",
        "hf_repo_id": "DiBiay/spec-fastgs-config-false-synthetic-specular-result",
        "log_name": "synthetic_specular_specfastgs_config_false_run.log",
        "zip_name": "spec_fastgs_config_false_output_synthetic_specular.zip",
    },
    {
        "name": "spec-fastgs-config-true",
        "display_name": "Spec-FastGS (config=True / Proposed Model)",
        "repo_subdir": "spec-fastgs",
        "script": "run_synthetic_specular_config_true.sh",
        "hf_repo_id": "DiBiay/spec-fastgs-config-true-synthetic-specular-result",
        "log_name": "synthetic_specular_specfastgs_config_true_run.log",
        "zip_name": "spec_fastgs_config_true_output_synthetic_specular.zip",
    },
]

venv_bin = os.path.dirname(sys.executable)
cuda_home = os.environ.get('CUDA_HOME', '/usr/local/cuda')
api = HfApi(token=HF_TOKEN)

all_results_summary = {}

for idx, exp in enumerate(EXPERIMENTS, start=1):
    disp = exp['display_name']
    repo_path = os.path.join(BASE_REPO_DIR, exp['repo_subdir'])
    script_path = os.path.join(repo_path, exp['script'])
    log_path = os.path.join(BASE_REPO_DIR, exp['log_name'])
    zip_path = os.path.join(BASE_REPO_DIR, exp['zip_name'])
    output_dir = os.path.join(repo_path, 'output', 'synthetic_specular')

    print(f"\n========================================================================")
    print(f" [{idx}/{len(EXPERIMENTS)}] RUNNING METHOD: {disp}")
    print(f"========================================================================")
    print(f" Repo Directory : {repo_path}")
    print(f" Batch Script   : {script_path}")
    print(f" HF Target Repo : {exp['hf_repo_id']}")
    print(f" Output Directory: {output_dir}")
    print(f"========================================================================\n")

    if not os.path.isfile(script_path):
        print(f"❌ ERROR: Batch script not found at {script_path}. Skipping.")
        continue

    # 1. Execute Batch Script
    cmd = f'''
export PATH={venv_bin}:{cuda_home}/bin:$PATH
export LD_LIBRARY_PATH={cuda_home}/lib64:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0
export DATA_ROOT={DATA_ROOT}
cd "{repo_path}"
bash {exp['script']} > "{log_path}" 2>&1
'''
    r_exec = subprocess.run(['bash', '-c', cmd])
    print(f"Execution finished with exit code: {r_exec.returncode}")
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            tail_lines = f.readlines()[-30:]
            print("--- Tail of Execution Log ---")
            print(''.join(tail_lines))

    # 2. Collect & Format Quantitative Results
    exp_metrics = {}
    res_file1 = os.path.join(output_dir, 'results.json')
    res_file2 = os.path.join(output_dir, 'results_grouped.json')
    target_res = res_file1 if os.path.isfile(res_file1) else (res_file2 if os.path.isfile(res_file2) else None)
    if target_res:
        with open(target_res, 'r') as f:
            exp_metrics = json.load(f)
        print(f"✅ Successfully loaded metrics from {target_res}")
    else:
        print(f"⚠️ Warning: Neither results.json nor results_grouped.json found in {output_dir}")

    all_results_summary[exp['name']] = {
        'display_name': disp,
        'metrics': exp_metrics,
        'hf_repo': exp['hf_repo_id']
    }

    # 3. Zip Output Archive
    if os.path.isdir(output_dir):
        print(f"\n📦 Archiving {output_dir} -> {zip_path} ...")
        shutil.make_archive(zip_path.replace('.zip', ''), 'zip', root_dir=output_dir)
        zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"Archive created: {zip_path} ({zip_size_mb:.2f} MB)")

        # 4. Upload to HuggingFace
        try:
            print(f"🚀 Uploading to HuggingFace repository: {exp['hf_repo_id']} ...")
            api.create_repo(repo_id=exp['hf_repo_id'], repo_type='dataset', exist_ok=True)
            api.upload_file(
                path_or_fileobj=zip_path,
                path_in_repo=exp['zip_name'],
                repo_id=exp['hf_repo_id'],
                repo_type='dataset'
            )
            if target_res:
                api.upload_file(
                    path_or_fileobj=target_res,
                    path_in_repo='summary_results.json',
                    repo_id=exp['hf_repo_id'],
                    repo_type='dataset'
                )
            print(f"🎉 Successfully uploaded {disp} outputs to HuggingFace!")
        except Exception as e_up:
            print(f"❌ HuggingFace upload failed for {disp}: {e_up}")

        # 5. Clean local disk space
        print(f"🧹 Cleaning up local output folder and zip archive to save server disk space...")
        try:
            os.remove(zip_path)
        except Exception:
            pass
        try:
            shutil.rmtree(output_dir)
        except Exception:
            pass
        print(f"Purged {output_dir} and {zip_path}.")
    else:
        print(f"⚠️ Output directory {output_dir} does not exist. Skipping zip and upload for {disp}.")


## c03 — Consolidated Master Quantitative Summary
Prints comparative table of all 5 methods on the Synthetic Specular dataset.

In [ ]:
# ── Consolidated Master Quantitative Table ─────────────────────────────────────
print("\n========================================================================================")
print("                     MASTER SYNTHETIC SPECULAR BENCHMARK SUMMARY TABLE                 ")
print("========================================================================================\n")

header = f"{ 'Method':<35s} | {'PSNR':<10s} | {'SSIM':<10s} | {'LPIPS':<10s} | {'HuggingFace Link':<45s}"
print(header)
print("-" * len(header))

for exp_name, info in all_results_summary.items():
    disp = info['display_name']
    metrics = info.get('metrics', {})
    hf_url = f"https://huggingface.co/datasets/{info['hf_repo']}"
    
    # Extract mean metrics if structured
    psnr_val = metrics.get('our_results', {}).get('SSIM', metrics.get('psnr', '-'))
    ssim_val = metrics.get('our_results', {}).get('SSIM', metrics.get('ssim', '-'))
    lpips_val = metrics.get('our_results', {}).get('LPIPS', metrics.get('lpips', '-'))
    
    print(f"{disp:<35s} | {str(psnr_val):<10s} | {str(ssim_val):<10s} | {str(lpips_val):<10s} | {hf_url:<45s}")

print("\n🎉 Benchmark sweep completed for all 5 methods!")